In [146]:
!pip install pulp
!pip install folium
!pip install pyproj

# **Seattle Collisions Project**

**Goal for the Project:** The goal for this project is to analyze the severity of collisions in Seattle to determine which intersections would benefit most from increased safety.

**For this Portion of the Code:** This does a basic analysis of intersections in Seattle, determining which intersections have had the most collisions, rather than checking for the severity of those collisions.

In [147]:
""" IMPORTS """

import numpy as np
import pandas as pd
import sqlite3
import os
import folium
from pyproj import Transformer
from pulp import LpMaximize, LpProblem, LpVariable, lpSum

""" CREATING PANDAS DATAFRAME AND SQL DATABASE """

df = pd.read_csv(r"SDOT_Collisions_All_Years.csv_1.xz", low_memory = False) # Creates dataframe from CSV
conn = sqlite3.connect(r"C:\Users\owenm\Jupyter Notebook Projects\Seattle Collisions Project\collisions.db") # Opens SQL connection

df["severity_score"] = np.where(
    df["PROPERTYDAMAGEONLY"] == "Y",
    1,
    (df["FATALITIES"] * 10) + (df["SERIOUSINJURIES"] * 5) + (df["INJURIES"] * 2)
) # This creates a new column in the dataframe df called "severity_score" with this biconditional

df_valid = df.dropna(subset = ["INTKEY"]).copy() # This gets rid of the null and NaN values and puts this in a new dataframe
df_valid["INTKEY"] = df_valid["INTKEY"].astype(int) # This filters all of the INTKEY column values to be type int

df_valid.to_sql(
    "collisions", # Name of table created inside SQL database
    conn, # Tells pandas which database file to connect to
    if_exists = "replace", # If a table named "collisions" already exists in that database, drop it and recreate it fresh
    index = False # Don't write pandas' internal row index (0, 1, 2, 3...) as its own column in the database
)

print("SQL Database Location: " + os.getcwd()) # Finds location of sql database

""" TOP LOCATIONS WITH MOST COLLISIONS """

num_of_intersections_considered = 10 # Presets the number of intersections that will be considered, can be altered at any time

intersections = (
    df_valid.groupby("INTKEY").size().sort_values(ascending = False).index
) # This creates a list of intersections by their INTKEY column - the ".index" makes sure that these are the indices, not the counts

if num_of_intersections_considered > len(intersections) or num_of_intersections_considered <= 0:
    num_of_intersections_considered = 10 # Sets as default in case the number considered doesn't work for the example

top_collision_table = (
    df_valid.groupby("INTKEY")
        .agg(
            TOTAL_COLLISIONS = ("severity_score", "count"),
            TOTAL_SEVERITY_SCORE = ("severity_score", "sum")
        )
        .reset_index()
        .sort_values("TOTAL_COLLISIONS", ascending = False)
        .head(num_of_intersections_considered)
) # Creates a table that includes the INTKEY, collision count, and severity score values for each of the top intersections sorted by collision count

print(f"\nTop { num_of_intersections_considered } locations with most collisions: \n")
print(top_collision_table.to_string(index = False))

conn.close() # This closes the SQL connection.

FileNotFoundError: [Errno 2] No such file or directory: 'SDOT_Collisions_All_Years.csv_1.xz'

## Checking for Severity of Collisions

**Methodology:** In the first example, the top intersections were based only on which locations had the most collisions. In this code, the column "severity_score" will be considered along with the amount of collisions to determine what the top intersections with the most severe collisions are, increasing the potential safety impact.

**PuLP:** PuLP comes with its own solver, which is used to design the optimization problem discussed and solve it.

In [ ]:
""" CONNECTING TO NEW SQL DATABASE """

severity_conn = sqlite3.connect(r"C:\Users\owenm\Jupyter Notebook Projects\Seattle Collisions Project\severity_of_collisions.db")
# Makes a connection to a new sql server with the "severity_score" column

df_valid.to_sql(
    "collisions", # Name of table created inside SQL database
    severity_conn, # Tells pandas which database file to connect to
    if_exists = "replace", # If a table named "collisions" already exists in that database, drop it and recreate it fresh
    index = False # Don't write pandas' internal row index (0, 1, 2, 3...) as its own column in the database
)

print("SQL Database Location: " + os.getcwd()) # Finds location of sql database

""" CREATING THE PROBLEM, VARIABLES, CONSTRAINTS, AND SOLVING IT """

model = LpProblem(
    "Seattle_Traffic_Safety",
    LpMaximize
) # This defines the specific problem trying to be optimized as "model", and specifies that the goal is to maximize it

all_intersections = {
    intersection: LpVariable(
        f"x_{intersection}",
        cat="Binary"
    )
    for intersection in intersections
} # This creates a dictionary of keys (each intersection) and values (binary category which is 0 or 1 depending on if intersection is selected)

df_inter = df_valid[df_valid["INTKEY"].isin(intersections)] # This creates a new dataframe limited to only the INTKEY values in intersections

model += lpSum(
    row["severity_score"] * all_intersections[row["INTKEY"]]
    for _, row in df_inter.iterrows()
) # This weighs the severity scores with each of the INTKEY values for each of the intersections in the df_inter dataframe and adds them

model += lpSum(all_intersections.values()) <= num_of_intersections_considered # Adds a constraint on the number of intersections considered in the model

model.solve() # This actively solves the model

selected = [
    intersection
    for intersection in all_intersections
    if all_intersections[intersection].value() == 1
] # This variable contains all of the selected values for optimizing the model

top_severity_table = (
    df_inter[df_inter["INTKEY"].isin(selected)]
        .groupby("INTKEY")
        .agg(
            TOTAL_COLLISIONS = ("severity_score", "count"),
            TOTAL_SEVERITY_SCORE = ("severity_score", "sum")
        )
        .reset_index()
        .sort_values("TOTAL_SEVERITY_SCORE", ascending = False)
) # Creates a table that includes the INTKEY, collision count, and severity score values for each of the top intersections sorted by severity

print(f"\nThese are the { num_of_intersections_considered } intersections that are most optimal for safety impact: \n")
print(top_severity_table.to_string(index = False))

severity_conn.close()

## Map Display

**Methodology:** This code below will display the geographical locations of each of these intersections in Seattle on a map using the Python tool folium. The user can use this as a visual indicator of the best locations for maximum safety impact. Only the data detailing the most severe collisions will be displayed.

In [ ]:
""" INTERSECTION COORDS AND CONVERSION """

# Washington State Plane North (feet) -> Standard latitude and longitude
transformer = Transformer.from_crs("EPSG:2926", "EPSG:4326", always_xy = True)

intersection_coords_most_severe = (
    df_inter[df_inter["INTKEY"].isin(selected)]
        .groupby("INTKEY")
        .agg(
            LAT = ("y", "mean"),
            LON = ("x", "mean"),
            TOTAL_COLLISIONS = ("severity_score", "count"),
            SEVERITY_SCORE = ("severity_score", "sum")
        )
        .reset_index()
) # The coords of each intersection for measuring which intersections have the most severe collisions

intersection_coords_most_severe["LON"], intersection_coords_most_severe["LAT"] = transformer.transform(
    intersection_coords_most_severe["LON"].values,
    intersection_coords_most_severe["LAT"].values
) # Converts all of the coordinates so that they are in correct locations

""" CREATING THE MAP """

map_center_most_severe = [47.6062, -122.3321]
map_most_severe = folium.Map(location = map_center_most_severe, zoom_start = 12)
# This centers the map in Seattle, and then creates the map

for _, row in intersection_coords_most_severe.iterrows():
    folium.Marker(
        location = [row["LAT"], row["LON"]],
        popup = f"INTKEY: {row['INTKEY']}<br>TOTAL COLLISIONS: {row['TOTAL_COLLISIONS']}<br>SEVERITY SCORE: {row['SEVERITY_SCORE']}",
        icon = folium.Icon(color = "blue", icon = "info-sign")
    ).add_to(map_most_severe)
# This marks red dots at each of the intersections on the map

map_most_severe.save("most_severe_collisions_map.html") # Saves the map as an html file

map_most_severe # This prints the map inline